# 01 - Exploring the KFC reviews
First look at the data, the problems I found while looking, and how each one led to a change in `src/kfc_absa/cleaning.py`.

Run `python scripts/run_clean.py` first to create `data/processed/reviews.csv`. Sections 4 and 5 also read the raw file in `data/raw/`, which is not in the repo (see the README).

In [ ]:
import collections

import pandas as pd

from kfc_absa.cleaning import detail_keys, normalize_text
from kfc_absa.config import RAW_PATH
from kfc_absa.data_io import load_raw_reviews, load_reviews

pd.set_option("display.max_colwidth", 90)
df = load_reviews()

## 1. Size and columns

In [ ]:
print(df.shape)
list(df.columns)

In [ ]:
df[["id", "rating", "iso_date", "clean_text"]].sample(10, random_state=42)

## 2. Ratings and missing values
Ratings are heavily skewed to 1 star. The aspect scores (`food`, `service`, `atmosphere`) and the details (`wait_time`, `order_type`...) are only filled when the reviewer chose to give them.

In [ ]:
df["rating"].value_counts().sort_index()

In [ ]:
df.isna().sum().sort_values(ascending=False)

## 3. Review length

In [ ]:
df["clean_text"].str.len().describe()

## 4. Problem found: `order_type` held the wrong data
The first version of the cleaning code filled `order_type` from the raw field `meal_type`, so Dine in / Take out / Delivery never reached the table. The raw file has both fields:

In [ ]:
if RAW_PATH.exists():
    raw = load_raw_reviews()
    print(f"{len(detail_keys(raw))} different keys inside 'details' across all reviews")
    for key in ["order_type", "meal_type"]:
        counts = collections.Counter((r.get("details") or {}).get(key) for r in raw)
        print(f"\n{key}:", dict(counts.most_common()))
else:
    print("Raw file not found, skipping (run scripts/run_scrape.py, or copy raw_reviews.json into data/raw/).")

Fixed in `flatten_reviews`: `order_type` and `meal_type` are now separate columns. The table agrees with the raw counts:

In [ ]:
print(df["order_type"].value_counts(dropna=False).to_dict())
print(df["meal_type"].value_counts(dropna=False).to_dict())

## 5. Problem found: messy text
Reviews contain line breaks, curly quotes and extra spaces. `normalize_text` adds a `clean_text` column and keeps the original `text`.
How many reviews changed, and what does a change look like?

In [ ]:
changed = df[df["text"] != df["clean_text"]]
print(f"{len(changed)} of {len(df)} reviews changed")
changed[["id", "text", "clean_text"]].head(3)

The function applied to the original text reproduces the saved column exactly, so the table and the code agree:

In [ ]:
(df["text"].map(normalize_text) == df["clean_text"]).all()

## 6. Hidden characters
`print` and tables hide characters you cannot see. `isprintable()` finds them, `Counter` counts them.

In [ ]:
found = collections.Counter(ch for text in df["text"] for ch in text if not ch.isprintable())
for ch, n in found.most_common():
    print(repr(ch), hex(ord(ch)), n)

Two kinds: the line break (handled by the whitespace rule) and one zero-width joiner (`U+200D`). That joiner is part of an emoji, so removing it splits the emoji apart. **To do:** keep it in `normalize_text`.

## Next
- Keep the emoji joiner, add language detection
- Tests for the cleaning functions
- Aspect-based sentiment analysis